# Module 7 Solution Guide: Dashboard, Trust, and Deployment Framing

Module 7 reframes the project as a business-facing AI system. The dashboard does not need to be a full production app, but it should show metrics, evidence, and limitations clearly.


## 1. Build Dashboard-Ready Metrics


In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/project_sec10k_rag")
DATA_DIR = BASE_DIR / "data"
OUTPUTS_DIR = DATA_DIR / "outputs"
REPORTS_DIR = BASE_DIR / "reports"

for folder in [DATA_DIR, OUTPUTS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

artifact_paths = {
    "download_manifest": OUTPUTS_DIR / "download_manifest.csv",
    "filings": OUTPUTS_DIR / "filing_metadata.csv",
    "extracted_items": OUTPUTS_DIR / "extracted_item_metadata.csv",
    "cleaned_items": OUTPUTS_DIR / "cleaned_item_metadata.csv",
    "chunks": DATA_DIR / "chunked" / "sec10k_chunks.parquet",
    "embedding_manifest": DATA_DIR / "vector_store" / "embedding_manifest.json",
    "rag_answers": OUTPUTS_DIR / "rag_answers.csv",
    "rag_evaluation": OUTPUTS_DIR / "rag_evaluation.csv",
    "market_features": OUTPUTS_DIR / "company_price_features.csv",
}

metrics = []
for metric_name, path in artifact_paths.items():
    metrics.append(
        {
            "metric_name": metric_name,
            "source_path": str(path.relative_to(BASE_DIR)),
            "available": path.exists(),
        }
    )

dashboard_metrics = pd.DataFrame(metrics)

manifest_path = artifact_paths["download_manifest"]
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    dashboard_metrics.loc[len(dashboard_metrics)] = {
        "metric_name": "companies_with_downloaded_10k",
        "source_path": "data/outputs/download_manifest.csv",
        "available": manifest.loc[manifest["download_status"].eq("downloaded"), "ticker"].nunique(),
    }

embedding_manifest_path = artifact_paths["embedding_manifest"]
if embedding_manifest_path.exists():
    embedding_manifest = json.loads(embedding_manifest_path.read_text(encoding="utf-8"))
    dashboard_metrics.loc[len(dashboard_metrics)] = {
        "metric_name": "embedding_backend",
        "source_path": "data/vector_store/embedding_manifest.json",
        "available": embedding_manifest.get("backend_used", ""),
    }

dashboard_metrics.to_csv(OUTPUTS_DIR / "dashboard_metrics.csv", index=False)
dashboard_metrics


## 2. Add Trust Flags

Trust flags help a stakeholder decide whether the output is ready to use. They should be simple, visible, and tied to evidence.


In [ ]:
trust_flags = pd.DataFrame(
    [
        {"area": "Retrieval", "green_flag": "Top chunks match the question", "red_flag": "Top chunks are unrelated"},
        {"area": "Generation", "green_flag": "Every claim has a citation", "red_flag": "Answer includes unsupported claims"},
        {"area": "Coverage", "green_flag": "Most companies and years are represented", "red_flag": "Many missing filings"},
        {"area": "Market analysis", "green_flag": "Price features connect to filing findings", "red_flag": "Market section is only descriptive"},
        {"area": "Deployment", "green_flag": "Limitations and refusal behavior are documented", "red_flag": "User sees answers without warnings"},
    ]
)

trust_flags.to_csv(OUTPUTS_DIR / "dashboard_trust_flags.csv", index=False)
trust_flags


## 3. Provenance Drill-Down Template

A dashboard should let a user move from a metric or answer back to the source evidence. This table gives the minimum fields for that drill-down.


In [ ]:
provenance_view = pd.DataFrame(
    [
        {
            "user_question": "",
            "answer_claim": "",
            "chunk_id": "",
            "ticker": "",
            "filing_year": "",
            "item_number": "",
            "source_file": "",
            "evidence_snippet": "",
        }
    ]
)

provenance_view.to_csv(OUTPUTS_DIR / "dashboard_provenance_template.csv", index=False)
provenance_view


## 4. Final Deployment Framing

In the final write-up, describe the system as decision support, not automated investment advice. A careful deployment plan should include human review, citation visibility, logging, model evaluation, and clear limits around missing or weak evidence.
